In [12]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

import pandas as pd

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7004.67it/s]


In [10]:

# carico i dataset
synonym_df = pd.read_csv("synonym_pairs_final.csv")
antonym_df = pd.read_csv("antonym_pairs_final.csv")
random_df = pd.read_csv("random_pairs_final.csv")


# creo un unico set con tutte le parole presenti nei tre esperimenti

all_words = set()

# sinonimi
all_words.update(synonym_df["word"])
all_words.update(synonym_df["synonym"])

# contrari
all_words.update(antonym_df["word"])
all_words.update(antonym_df["antonym"])

# random
all_words.update(random_df["word"])
all_words.update(random_df["random"])


print("Numero totale parole:", len(all_words))

Numero totale parole: 638


In [ ]:
#calcolo gli embedding una sola volta grazie a all_words, li metto in un dizionario
embeddings = {}

for word in all_words:
    
    embeddings[word] = model.encode(
        word,
        convert_to_tensor=True
    )


print("Embedding creati:", len(embeddings))

Embedding creati: 638


In [19]:
#salvo il dizionario così da non doverli calcolare ogni volta che apro il progetto

import pickle

with open("embeddings.pkl", "wb") as f:
    pickle.dump(embeddings, f)

In [ ]:
#codice da eseguire per avere gli embedding

import pickle

with open("embeddings.pkl", "rb") as f:
    embeddings = pickle.load(f)

In [13]:
#funzione per la cosine similarity

def add_cosine_similarity(df, col1, col2):
    #in input entrano: il database delle coppie, la colonna della parola e la colonna della parola associata
    #si crea una lista  per i risultati della cosine similarity
    #si calcola la cosine similarity usando gli embedding già calcolati
    #si crea una colonna nel dataset di partenza con i risultati della cosine similarity
    
    similarities = []

    for w1, w2 in zip(df[col1], df[col2]):

        emb1 = embeddings[w1]
        emb2 = embeddings[w2]

        similarity = cos_sim(
            emb1,
            emb2
        ).item()

        similarities.append(similarity)


    df["cosine_similarity"] = similarities

    return df

In [15]:
#funzione per sinonimi

syn_df = add_cosine_similarity(
    synonym_df,
    "word",
    "synonym"
)

print(syn_df.head())

       word      synonym relation  cosine_similarity
0  revolved      rotated  synonym           0.397232
1   ghastly      macabre  synonym           0.251190
2     dodgy        dicey  synonym           0.389690
3  biweekly  fortnightly  synonym           0.378817
4   pouring      gushing  synonym           0.522623


In [16]:
#funzione per contrari
ant_df = add_cosine_similarity(
    antonym_df,
    "word",
    "antonym"
)

print(ant_df.head())

          word        antonym relation  cosine_similarity
0    ascending     descending  antonym           0.814616
1  centralized  decentralized  antonym           0.613580
2   reasonable   unreasonable  antonym           0.696652
3       simple        complex  antonym           0.469160
4      legible      illegible  antonym           0.461158


In [17]:
rand_df = add_cosine_similarity(
    random_df,
    "word",
    "random"
)

print(rand_df.head())

          word       random relation  cosine_similarity
0     plodding    bombastic   random           0.242246
1  unpalatable  melancholic   random           0.300340
2     obliging     definite   random           0.234148
3    punishing  recombinant   random           0.151135
4      opposed        ionic   random           0.161461


In [18]:
#salvo i risultati
syn_df.to_csv(
    "results_synonyms_similarity.csv",
    index=False
)

ant_df.to_csv(
    "results_antonyms_similarity.csv",
    index=False
)

rand_df.to_csv(
    "results_random_similarity.csv",
    index=False
)